langchain1.2的长期记忆是基于Store来持久化，相关的API有put()写入 get()获取 search()搜索

API PUT()的实现

基于InmemoryStore()的实现

In [2]:
from langgraph.store.memory import InMemoryStore
store = InMemoryStore()
namespace=("user")
user_id="user-1"
user_name="小小明"
store.put(namespace, user_id, {"name":user_name})
item = store.get(namespace, user_id)
print(item)


Item(namespace=['u', 's', 'e', 'r'], key='user-1', value={'name': '小小明'}, created_at='2026-09-19T15:42:41.651374+00:00', updated_at='2026-09-19T15:42:41.651382+00:00')


对ITEM进行更新，实际是1次覆盖

In [3]:
user_name="小小红"
store.put(namespace, user_id, {"name":user_name})
item = store.get(namespace, user_id)
print(item)

Item(namespace=['u', 's', 'e', 'r'], key='user-1', value={'name': '小小红'}, created_at='2026-09-19T15:43:37.352653+00:00', updated_at='2026-09-19T15:43:37.352654+00:00')


基于PostgreSQL的实现

In [4]:
from langgraph.types import Command
from dataclasses import dataclass
from typing import Literal, TypedDict, Annotated, Union

from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware, PIIMiddleware,   ModelCallLimitMiddleware,ModelFallbackMiddleware,LLMToolSelectorMiddleware,ContextEditingMiddleware,ClearToolUsesEdit,FilesystemFileSearchMiddleware
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy,    MultipleStructuredOutputsError, StructuredOutputValidationError
from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnableConfig
from dotenv import load_dotenv
from langchain.agents import create_agent
import os
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage,ToolMessage

from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import before_agent,before_model,after_agent,after_model
from typing import  Any,Callable
from langgraph.runtime import Runtime
from langchain.agents import AgentState
from langchain.agents.middleware import ModelCallResult,AgentMiddleware,wrap_model_call,ModelResponse,ModelRequest,wrap_tool_call,ToolCallRequest
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.store.postgres import PostgresStore
load_dotenv(override=True)
my_api_key = os.getenv("API_KEY")
my_base_url = os.getenv("BASE_URL")
#1 初始化大模型
myModel = init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=300,
    max_retries=3,
    # reasoning_effort="max"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)
#3 创建PostgreSQL数据库连接
DB_URL = os.getenv("DB_URL")
with PostgresStore.from_conn_string(DB_URL) as store:
    #初始化数据库
    store.setup()

    namespace=("user")
    user_id="user-1"
    user_name="小小明"
    store.put(namespace, user_id, {"name":user_name})
    item = store.get(namespace, user_id)
    print(item)

Item(namespace=['u', 's', 'e', 'r'], key='user-1', value={'name': '小小明'}, created_at='2026-09-19T23:46:40.409598+08:00', updated_at='2026-09-19T23:46:40.409598+08:00')


基于PostgreSQL的更新操作,不再是覆盖操作，而是对value的真正的更新

支持2种检索方式 1 安装filter做结构化的过滤，即使用value的键值做符合筛选条件的过滤 2 query 做语义相似度检索，需要把输入转换为向量 返回的都是匹配的SearchItem列表[Item1,Item2,...]，并额外携带匹配分数等检索元信息

API search()的实现

基于InmemeryStore的实现 按照namespace前缀搜索

In [11]:
from langgraph.store.memory import InMemoryStore
store = InMemoryStore()
namespace1=("users","Alice","memories")
key1 = 'preferences'
value1 = {
    "course": "计算机组成原理",
    "sports": "跑步",
    "food": "紫光园奶皮子酸奶"
}

namespace2 = ("users", "Bob", "memories")
key2 = 'preferences'
value2 = {
    "course": "数字电路与模拟电路",
    "sports": "跑步",
    "food": "奶皮子糖葫芦"
}

namespace3 = ("users", "Black", "memories")
key3 = 'preferences'
value3 = {
    "course": "数字电路与模拟电路",
    "sports": "羽毛球",
    "food": "紫光园奶皮子酸奶"
}


store.put(namespace1, value1, value1)
store.put(namespace2, value2, value2)
store.put(namespace3, value3, value3)
#按照namespace前缀搜索
for item in store.search(("users","Alice")):
    print(item)


Item(namespace=['users', 'Alice', 'memories'], key="{'course': '计算机组成原理', 'sports': '跑步', 'food': '紫光园奶皮子酸奶'}", value={'course': '计算机组成原理', 'sports': '跑步', 'food': '紫光园奶皮子酸奶'}, created_at='2026-09-19T16:02:18.292156+00:00', updated_at='2026-09-19T16:02:18.292159+00:00', score=None)


按照filter过滤

In [12]:
#按照namespace前缀搜索
for item in store.search(("users",),filter={"food": "紫光园奶皮子酸奶"}):
    print(item)


Item(namespace=['users', 'Alice', 'memories'], key="{'course': '计算机组成原理', 'sports': '跑步', 'food': '紫光园奶皮子酸奶'}", value={'course': '计算机组成原理', 'sports': '跑步', 'food': '紫光园奶皮子酸奶'}, created_at='2026-09-19T16:02:18.292156+00:00', updated_at='2026-09-19T16:02:18.292159+00:00', score=None)
Item(namespace=['users', 'Black', 'memories'], key="{'course': '数字电路与模拟电路', 'sports': '羽毛球', 'food': '紫光园奶皮子酸奶'}", value={'course': '数字电路与模拟电路', 'sports': '羽毛球', 'food': '紫光园奶皮子酸奶'}, created_at='2026-09-19T16:02:18.292242+00:00', updated_at='2026-09-19T16:02:18.292243+00:00', score=None)


按照语义过滤 使用自定义函数做嵌入

In [13]:
from langgraph.store.memory import InMemoryStore
from pprint import pprint as rprint
store = InMemoryStore()
#自定义1个嵌入函数
def embed(text:list[str])->list[float]:
    return [[1.0]*6 for _ in range(len(text))]
index_config={
    "embed":embed,
    "dims":6,
    "fields":["$","course"]
}

store = InMemoryStore(
    index=index_config
)
namespace1=("users","Alice","memories")
key1 = 'preferences'
value1 = {
    "course": "计算机组成原理",
    "sports": "跑步",
    "food": "紫光园奶皮子酸奶"
}

namespace2 = ("users", "Bob", "memories")
key2 = 'preferences'
value2 = {
    "course": "数字电路与模拟电路",
    "sports": "跑步",
    "food": "奶皮子糖葫芦"
}

namespace3 = ("users", "Black", "memories")
key3 = 'preferences'
value3 = {
    "course": "数字电路与模拟电路",
    "sports": "羽毛球",
    "food": "紫光园奶皮子酸奶"
}


store.put(namespace1, value1, value1)
store.put(namespace2, value2, value2)
store.put(namespace3, value3, value3)
#按照namespace前缀搜索
rprint(store._vectors)


defaultdict(<function InMemoryStore.__init__.<locals>.<lambda> at 0x00000189E313EA20>,
            {('users', 'Alice', 'memories'): defaultdict(<class 'dict'>,
                                                         {"{'course': '计算机组成原理', 'sports': '跑步', 'food': '紫光园奶皮子酸奶'}": {'$': [1.0,
                                                                                                                              1.0,
                                                                                                                              1.0,
                                                                                                                              1.0,
                                                                                                                              1.0,
                                                                                                                              1.0],
                                                     

按照语义过滤 使用嵌入模型做嵌入

In [14]:

from langgraph.types import Command
from dataclasses import dataclass
from typing import Literal, TypedDict, Annotated, Union

from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware, PIIMiddleware,   ModelCallLimitMiddleware,ModelFallbackMiddleware,LLMToolSelectorMiddleware,ContextEditingMiddleware,ClearToolUsesEdit,FilesystemFileSearchMiddleware
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy,    MultipleStructuredOutputsError, StructuredOutputValidationError
from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnableConfig
from dotenv import load_dotenv
from langchain.agents import create_agent
import os
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage,ToolMessage

from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import before_agent,before_model,after_agent,after_model
from typing import  Any,Callable
from langgraph.runtime import Runtime
from langchain.agents import AgentState
from langchain.agents.middleware import ModelCallResult,AgentMiddleware,wrap_model_call,ModelResponse,ModelRequest,wrap_tool_call,ToolCallRequest
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.store.postgres import PostgresStore
load_dotenv(override=True)
my_api_key = os.getenv("API_KEY")
my_base_url = os.getenv("BASE_URL")
#1 初始化大模型
myModel = init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=300,
    max_retries=3,
    # reasoning_effort="max"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)
myEmbeddingModel = init_chat_model(
    model="qwen3.7-text-embedding-flash",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    # reasoning_effort="max"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)

index_config={
    "embed":myEmbeddingModel,
    "dims":.2560,
    "fields":["$","course"]
}
store = InMemoryStore(
    index=index_config
)
namespace1=("users","Alice","memories")
key1 = 'preferences'
value1 = {
    "course": "计算机组成原理",
    "sports": "跑步",
    "food": "紫光园奶皮子酸奶"
}

namespace2 = ("users", "Bob", "memories")
key2 = 'preferences'
value2 = {
    "course": "数字电路与模拟电路",
    "sports": "跑步",
    "food": "奶皮子糖葫芦"
}

namespace3 = ("users", "Black", "memories")
key3 = 'preferences'
value3 = {
    "course": "数字电路与模拟电路",
    "sports": "羽毛球",
    "food": "紫光园奶皮子酸奶"
}


store.put(namespace1, value1, value1)
store.put(namespace2, value2, value2)
store.put(namespace3, value3, value3)
#按照namespace前缀搜索
rprint(store._vectors)

TypeError: 'ChatOpenAI' object is not callable

基于PostgreStore的实现

In [ ]:
from langgraph.types import Command
from dataclasses import dataclass
from typing import Literal, TypedDict, Annotated, Union

from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware, PIIMiddleware,   ModelCallLimitMiddleware,ModelFallbackMiddleware,LLMToolSelectorMiddleware,ContextEditingMiddleware,ClearToolUsesEdit,FilesystemFileSearchMiddleware
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy,    MultipleStructuredOutputsError, StructuredOutputValidationError
from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnableConfig
from dotenv import load_dotenv
from langchain.agents import create_agent
import os
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage,ToolMessage

from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import before_agent,before_model,after_agent,after_model
from typing import  Any,Callable
from langgraph.runtime import Runtime
from langchain.agents import AgentState
from langchain.agents.middleware import ModelCallResult,AgentMiddleware,wrap_model_call,ModelResponse,ModelRequest,wrap_tool_call,ToolCallRequest
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.store.postgres import PostgresStore
load_dotenv(override=True)
my_api_key = os.getenv("API_KEY")
my_base_url = os.getenv("BASE_URL")
#1 初始化大模型
myModel = init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=300,
    max_retries=3,
    # reasoning_effort="max"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)
#3 创建PostgreSQL数据库连接
DB_URL = os.getenv("DB_URL")
with PostgresStore.from_conn_string(DB_URL) as store:
    #初始化数据库
    store.setup()

    namespace=("user")
    user_id="user-1"
    user_name="小小明"
    store.put(namespace, user_id, {"name":user_name})
    item = store.get(namespace, user_id)
    print(item)